# Knowledge Distillation
"Distilling the Knowledge in a Neural Network" (Hinton, Vinyals, Dean, 2015)

**Idea:** a big trained network (the **teacher**) knows more than the labels say. For an image of a 2 it outputs not just "2" but also "a bit like a 3, a bit like a 7, nothing like a 4". Train a small network (the **student**) to match these **soft** probabilities, softened further with a temperature, and it learns far more per example than from the hard label alone.

### Ways to get a small model
| | Train small from scratch | Distillation | Pruning | Quantization | LoRA (`lora.ipynb`) |
|---|---|---|---|---|---|
| What changes | - | a **new, smaller** network imitates the big one | weights of the big one removed | weights stored in fewer bits | small low-rank update added |
| Training signal | hard labels | teacher's **soft targets** (+ hard labels) | - | - | hard labels |
| Needs the teacher's weights? | - | no, only its outputs | yes | yes | yes |
| Architecture of result | any | **any** | same, sparser | same | same |
| Needs labels? | yes | **no** (teacher labels any input) | - | - | yes |
| Goal | - | cheaper inference | cheaper inference | cheaper inference | cheaper fine-tuning |

Here: the paper's MNIST setup. A large teacher MLP, a small student, the student trained with and without distillation, and the paper's striking experiment: a student that has **never seen a 3** learns to recognize 3s from the teacher's soft targets on other digits.

Flow: `x (B, 784) -> teacher logits v (B, 10) -> softmax(v / T) = soft targets` and `x -> student logits z (B, 10) -> loss = alpha * T^2 * KL(soft targets || softmax(z / T)) + (1 - alpha) * CE(z, y)`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - MNIST
60k train / 10k test, flattened and normalized, held on the GPU. As in the paper, training images are **jittered** by up to 2 pixels in each direction for the teacher (cheap augmentation that makes it generalize better); the student sees the plain training set, which the paper calls the **transfer set**.

In [ ]:
train_ds = datasets.MNIST("../data", train=True, download=True)
test_ds = datasets.MNIST("../data", train=False, download=True)

def prep(ds):
    x = (ds.data.float() / 255 - 0.1307) / 0.3081                      # (N, 28, 28)
    return x.to(device), ds.targets.to(device)

Xtr, ytr = prep(train_ds)        # (60000, 28, 28), (60000,)
Xte, yte = prep(test_ds)         # (10000, 28, 28), (10000,)
pad_value = (0 - 0.1307) / 0.3081                                       # a black pixel after normalization

def jitter(x, max_shift=2):
    # (B, 28, 28) -> (B, 28, 28), each image shifted by up to 2 px in x and y
    B = x.shape[0]
    xp = F.pad(x, (max_shift,) * 4, value=pad_value)                    # (B, 32, 32)
    dx, dy = torch.randint(0, 2 * max_shift + 1, (2, B), device=x.device)
    idx = torch.arange(28, device=x.device)
    rows = (dy[:, None] + idx)[:, :, None]                              # (B, 28, 1)
    cols = (dx[:, None] + idx)[:, None, :]                              # (B, 1, 28)
    return xp[torch.arange(B, device=x.device)[:, None, None], rows, cols]

print(Xtr.shape, Xte.shape, jitter(Xtr[:4]).shape)

## 2. Teacher and student
- **Teacher**: `784 -> 1200 -> 1200 -> 10` with dropout (the paper's "cumbersome" model).
- **Student**: `784 -> 800 -> 800 -> 10`, **no** dropout, no jitter (the paper's student). About half the parameters; the bigger handicap is the missing regularization and augmentation, which the soft targets are meant to make up for.

Paper numbers for this pair: teacher 67 test errors, student alone 146, distilled student 74.

In [ ]:
class MLP(nn.Module):
    # (B, 28, 28) -> logits (B, 10)
    def __init__(self, hidden, depth, dropout=0.0):
        super().__init__()
        layers, d = [], 784
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout)]
            d = hidden
        self.net = nn.Sequential(*layers, nn.Linear(d, 10))

    def forward(self, x):
        return self.net(x.flatten(1))

def make_teacher():
    return MLP(hidden=1200, depth=2, dropout=0.3).to(device)

def make_student(seed=0):
    torch.manual_seed(seed)
    return MLP(hidden=800, depth=2).to(device)

count = lambda m: sum(p.numel() for p in m.parameters())
print(f"teacher params: {count(make_teacher()) / 1e6:.2f}M | student params: {count(make_student()) / 1e6:.2f}M")

@torch.no_grad()
def errors(model, X=Xte, y=yte):
    model.eval()
    return (model(X).argmax(-1) != y).sum().item()

## 3. Train the teacher
Ordinary supervised training: cross-entropy on hard labels, Adam with cosine decay, jittered inputs, 20 epochs.

In [ ]:
def train(model, loss_fn, epochs, X=Xtr, y=ytr, augment=False, lr=1e-3, batch_size=128):
    # loss_fn(student_logits, x, y) -> scalar
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * math.ceil(len(X) / batch_size))
    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(len(X), device=device)
        for i in range(0, len(X), batch_size):
            ix = perm[i:i + batch_size]
            xb = jitter(X[ix]) if augment else X[ix]
            loss = loss_fn(model(xb), xb, y[ix])
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
    return model

hard_loss = lambda logits, x, y: F.cross_entropy(logits, y)

torch.manual_seed(42)
teacher = train(make_teacher(), hard_loss, epochs=20, augment=True)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)
print(f"teacher test errors: {errors(teacher)} / 10000")

## 4. Soft targets and temperature
A well-trained teacher is very confident: on a typical 2 it gives $p(2) \approx 0.999$ and the rest shares 0.001. The interesting information (which wrong classes are *less* wrong) is in the ratios of those tiny probabilities, where it has almost no effect on a cross-entropy loss.

Dividing the logits by a **temperature** $T > 1$ before the softmax flattens the distribution and brings that structure out. Hinton calls it the teacher's "dark knowledge".

### Formula: Softmax with temperature
$$p_i^{(T)} = \frac{\exp(v_i / T)}{\sum_j \exp(v_j / T)}$$
- $v \in \mathbb{R}^{10}$: teacher logits
- $T = 1$: the ordinary softmax; $T \to \infty$: uniform; $T \to 0$: one-hot argmax
- the same $T$ is applied to the student's logits $z$ during distillation; at test time the student uses $T = 1$

In [ ]:
with torch.no_grad():
    logits = teacher(Xte)                                                # (10000, 10)
i = (yte == 2).nonzero()[0].item()                                       # first test image of a 2
torch.set_printoptions(precision=4, sci_mode=False)
for T in [1, 4, 10]:
    print(f"T = {T:2d}:", (logits[i] / T).softmax(-1).cpu())

fig, ax = plt.subplots(1, 4, figsize=(14, 2.8))
ax[0].imshow(Xte[i].cpu(), cmap="gray"); ax[0].axis("off")
for a, T in zip(ax[1:], [1, 4, 10]):
    a.bar(range(10), (logits[i] / T).softmax(-1).cpu()); a.set_xticks(range(10)); a.set_title(f"teacher, T = {T}"); a.set_ylim(0, 1)
plt.show()

# averaged over all test images of each digit: which other digits does the teacher find similar?
sim = torch.stack([(logits[yte == d] / 4).softmax(-1).mean(0) for d in range(10)]).cpu()   # (10, 10)
sim.fill_diagonal_(0)
plt.figure(figsize=(4.5, 4))
plt.imshow(sim, cmap="viridis"); plt.xlabel("teacher's soft probability for digit"); plt.ylabel("true digit")
plt.xticks(range(10)); plt.yticks(range(10)); plt.title("off-diagonal soft targets, T = 4"); plt.colorbar(); plt.show()

## 5. The distillation loss
The student is trained on a weighted sum of two terms: match the teacher's softened distribution, and (optionally) still get the true label right.

### Formula: Distillation loss
$$\mathcal{L} = \alpha\, T^2\, D_{KL}\big(p^{(T)}\ \|\ q^{(T)}\big) + (1 - \alpha)\, \text{CE}\big(y,\ q^{(1)}\big)$$
- $p^{(T)} = \text{softmax}(v / T)$: teacher's soft targets; $q^{(T)} = \text{softmax}(z / T)$: student's softened prediction
- $q^{(1)}$: student's ordinary prediction; $y$: the true label
- $\alpha$: weight of the soft term (0.9 here; the paper found a small weight on the hard term works best)
- KL and cross-entropy against $p^{(T)}$ differ only by the teacher's entropy, a constant for the student

### Formula: Why the $T^2$
$$\frac{\partial}{\partial z_i}\, D_{KL}\big(p^{(T)}\ \|\ q^{(T)}\big) = \frac{1}{T}\big(q_i^{(T)} - p_i^{(T)}\big) \approx \frac{1}{N T^2}\,(z_i - v_i) \quad \text{(high } T \text{, zero-mean logits)}$$
- $N = 10$ classes
- the soft-target gradient shrinks as $1 / T^2$; multiplying by $T^2$ keeps the two terms' relative weight the same when $T$ changes
- the high-$T$ limit also shows that distillation contains **logit matching** (minimizing $\frac{1}{2}(z_i - v_i)^2$) as a special case

In [ ]:
def distill_loss_fn(T, alpha):
    def loss_fn(student_logits, x, y):
        with torch.no_grad():
            soft = (teacher(x) / T).softmax(-1)                          # (B, 10) teacher soft targets
        log_q = F.log_softmax(student_logits / T, dim=-1)                # (B, 10)
        kd = F.kl_div(log_q, soft, reduction="batchmean") * T * T        # scalar
        return alpha * kd + (1 - alpha) * F.cross_entropy(student_logits, y)
    return loss_fn

## 6. Student: hard labels vs distillation
Same student, same initialization, same data and schedule (10 epochs). The only difference is the loss. A temperature sweep shows the trade-off: too low and the soft targets are nearly one-hot, too high and they are nearly uniform. The paper used $T = 20$ for the 800-unit student and found 2.5-4 best when the student is shrunk to 30 units per layer: a small student cannot afford to model the teacher's very negative logits.

In [ ]:
student_epochs = 10

baseline = train(make_student(), hard_loss, student_epochs)
base_err = errors(baseline)
print(f"teacher                    : {errors(teacher):4d} test errors")
print(f"student, hard labels       : {base_err:4d} test errors")

results = {}
for T in [1, 2, 4, 8, 20]:
    s = train(make_student(), distill_loss_fn(T, alpha=0.9), student_epochs)
    results[T] = errors(s)
    print(f"student, distilled (T = {T:2d}) : {results[T]:4d} test errors")

plt.figure(figsize=(5, 3))
plt.plot(list(results), list(results.values()), marker="o", label="distilled")
plt.axhline(base_err, ls="--", c="gray", label="hard labels"); plt.axhline(errors(teacher), ls=":", c="C1", label="teacher")
plt.xscale("log", base=2); plt.xlabel("temperature T"); plt.ylabel("test errors"); plt.legend(); plt.show()

## 7. Learning a digit that was never shown
Remove **every 3** from the transfer set. The student never sees a 3 or the label "3".
- trained on hard labels, it cannot ever output 3: it gets all ~1010 test 3s wrong
- trained on the teacher's soft targets with **no hard-label term** ($\alpha = 1$), it still sees "this 8 is a little 3-like, this 5 is a little 3-like", which is enough to locate the class

The paper reports 98.6% of test 3s right after correcting the bias of the 3 output (which is too low, since 3 was never the top class in training). We do the same correction: add a constant (the paper's 3.5) to the 3 logit.

In [ ]:
keep = ytr != 3
X_no3, y_no3 = Xtr[keep], ytr[keep]
is3 = yte == 3
print(f"transfer set without 3s: {len(X_no3)} images | test 3s: {is3.sum().item()}")

@torch.no_grad()
def report(name, model, bias3=0.0):
    model.eval()
    logits = model(Xte)                                                  # (10000, 10)
    logits[:, 3] += bias3
    pred = logits.argmax(-1)
    print(f"{name:42s} | total errors {(pred != yte).sum().item():5d} | 3s correct {(pred[is3] == 3).sum().item():4d} / {is3.sum().item()}")

hard_no3 = train(make_student(), hard_loss, student_epochs, X=X_no3, y=y_no3)
soft_no3 = train(make_student(), distill_loss_fn(T=8, alpha=1.0), student_epochs, X=X_no3, y=y_no3)

report("hard labels, no 3s", hard_no3)
report("distilled, no 3s", soft_no3)
report("distilled, no 3s, bias of class 3 + 3.5", soft_no3, bias3=3.5)

## Notes
- Soft targets carry much more information per example than a hard label (a full distribution instead of one of $N$ choices) and have much lower gradient variance between examples. That is why the student can be trained on less data, without regularization, and even with classes missing.
- The transfer set does not need labels at all ($\alpha = 1$): the teacher can label unlabeled or even synthetic inputs. This is the form used for LLMs: generate data with a strong model, train a small one on it (`models.md`, post-training). With an API-only teacher there are no logits, so it reduces to training on the teacher's sampled text ("hard" distillation).
- The teacher can be an **ensemble**: average the members' softened probabilities and distill them into one network. That was the paper's motivating use (and its speech-recognition result).
- Known results: **DistilBERT** (6 layers, 97% of BERT's GLUE score, 40% smaller, 60% faster) and TinyBERT / MobileBERT also match intermediate features and attention maps, not just the output distribution.
- Distillation is not a guaranteed win. How much it helps depends on the capacity gap, $T$, $\alpha$ and the training budget; a very small student (e.g. `784 -> 30 -> 10`) can do *worse* with soft targets than with hard labels because it spends its capacity imitating details it cannot represent. Treat §6 as an experiment to run, not a fixed result.
- A higher temperature is not always better: at high $T$ the student must also match the teacher's very negative logits, which are mostly noise, and a small student wastes capacity on them.
- Distillation also works with a teacher of the *same* size (self-distillation, "born-again networks") and usually improves on it: the soft targets act as a learned, per-example **label smoothing** (`transformer.ipynb`).